In [ ]:
# ! pip install google-cloud-aiplatform vertexai

In [2]:
from typing import Sequence
import dataclasses
import json
from vertexai import generative_models

GENERATION_AUTORATER_EVAL_PROMPT_WITH_IMAGES = """
You are an evaluation expert. You will be provided with a query, a generated response and a ground truth. Your task is to analyze the generated response for accuracy, completeness, and relevance compared to the ground truth.
You may also be given with images that provided by generated response or ground truth, in this case, you also need to evaluate if the generated response images are relevant to the ground truth results.

Please provide:
A float score from 0 to 5, where 0 is completely inaccurate and 5 is perfectly accurate.
A detailed explanation of why you gave that score, highlighting specific areas where the generated response was correct, incorrect, missing information, or irrelevant.
The output should be in JSON format.

EXAMPLE:

Query: What's the Capital of China
Generated Response: Victoria.
Ground Truth: Beijing

EVALUATION
{
"score" : 0.0,
"reason" : "The generated response is incorrect based on provided ground truth."
}
"""

RESPONSE_SCHEMA_SCORE = {
    "type": "object",
    "properties": {
        "score": {"type": "number", "minimum": 0, "maximum": 5},
        "reason": {"type": "string"},
    },
    "required": ["score", "reason"],
}


@dataclasses.dataclass(frozen=True)
class GenerationEvaluation:
  """The generation evaluation result.

  Attributes:
    score: The score of the generation evaluation.
    reason: The reason of the generation evaluation.
  """
  score: float
  reason: str


def eval_generation(
    autorater_model: generative_models.GenerativeModel,
    question: Sequence[generative_models.Part],
    model_reply: Sequence[generative_models.Part],
    ground_truth: Sequence[generative_models.Part],
) -> GenerationEvaluation:
  """Evaluates the generated response for accuracy, completeness, and relevance compared to the ground truth.

  Args:
    autorater_model: The autorater model to use.
    question: The question to ask the model.
    model_reply: The model's reply to the question.
    ground_truth: The ground truth answer to the question.

  Returns:
    The generation evaluation result.
  """
  part_list = []
  part_list.append(
      generative_models.Part.from_text(
          GENERATION_AUTORATER_EVAL_PROMPT_WITH_IMAGES
      )
  )

  part_list.append(generative_models.Part.from_text("\nQuery: "))
  for part in question:
    part_list.append(part)

  part_list.append(generative_models.Part.from_text("\nGenerated Response: "))
  for part in model_reply:
    part_list.append(part)

  part_list.append(generative_models.Part.from_text("\nGround Truth: "))
  for part in ground_truth:
    part_list.append(part)

  response: generative_models.GenerationResponse = (
      autorater_model.generate_content(
          part_list,
          generation_config=generative_models.GenerationConfig(
              temperature=0,
              response_mime_type="application/json",
              response_schema=RESPONSE_SCHEMA_SCORE,
          ),
      )
  )
  score_eval_response_dict = json.loads(response.candidates[0].text)
  eval_result = GenerationEvaluation(
      score=score_eval_response_dict["score"],
      reason=score_eval_response_dict["reason"],
  )
  return eval_result

In [26]:
import vertexai
vertexai.init(project="ivanmkc-experimental-2-631260")

In [27]:
import asyncio
from async_lru import alru_cache

model_name = "gemini-2.5-flash"
model: generative_models.GenerativeModel = generative_models.GenerativeModel(
    model_name=model_name,
)

question = "What is the capital of Canada"
model_reply = "Obviously, Toronto"
ground_truth = "Ottawa"

@alru_cache(maxsize=None)
async def eval_generation_async(question: str, model_reply: str, ground_truth: str, semaphore: asyncio.Semaphore) -> GenerationEvaluation:
    async with semaphore:
        return await asyncio.to_thread(eval_generation,
            autorater_model=model,
            question=[generative_models.Part.from_text(question)],
            model_reply=[generative_models.Part.from_text(model_reply)],
            ground_truth=[generative_models.Part.from_text(ground_truth)]
        )

In [28]:
# corpus = [
#     # 1
#     ("What is the capital of Canada?", "Obviously, Toronto.", "Ottawa"),

#     # 2
#     ("Which planet is closest to the Sun?", "That would be Venus, the hottest planet.", "Mercury"),

#     # 3
#     ("Who wrote the novel '1984'?", "I'm pretty sure that was Aldous Huxley.", "George Orwell"),

#     # 4
#     ("What is the tallest mountain in the world?", "Mount Everest, without a doubt.", "Mount Everest"),

#     # 5
#     ("Who painted the Mona Lisa?", "The famous Michelangelo, of course.", "Leonardo da Vinci"),

#     # 6
#     ("What is the largest mammal on Earth?", "It's the African Elephant.", "Blue Whale"),

#     # 7
#     ("What is the chemical symbol for gold?", "Easy, that's Ag.", "Au"),

#     # 8
#     ("In which city is the Golden Gate Bridge located?", "That's in Los Angeles.", "San Francisco"),

#     # 9
#     ("What is the longest river in the world?", "It has always been the Nile River.", "The Amazon River"),

#     # 10
#     ("Who was the first President of the United States?", "Thomas Jefferson, who wrote the Declaration of Independence.", "George Washington")
# ]

In [29]:
# semaphore = asyncio.Semaphore(10)
# results = await asyncio.gather(*[eval_generation_async(question=question, model_reply=model_reply, ground_truth=ground_truth, semaphore=semaphore) 
#                                 for (question, model_reply, ground_truth) in corpus]
#                                 )

In [30]:
# results

In [31]:
from llm_auditor.claims import Claim, QueryAmbiguity
import yaml
from pathlib import Path
from dataclasses import asdict

claims_dir_path = Path("claims")

# Read the YAML file
with open(claims_dir_path / "claims_with_queries.yaml", "r") as f:
    claims_from_yaml = yaml.safe_load(f)

# Convert the list of dictionaries back to a list of Claim objects
claims = [Claim(**claim_data) for claim_data in claims_from_yaml]

In [32]:
# import pandas as pd

# # Load claims from cache
# df = pd.read_csv("critic_output.csv")

In [33]:
# df.head()

In [34]:
# claims_cached = df.to_dict(orient='records')
# claims_cached[0]

# Claims need a corresponding query


In [35]:
# import dspy
# import mlflow
# mlflow.dspy.autolog()
# mlflow.set_experiment("generate_queries")

# model_name = "gemini/gemini-2.5-pro-preview-06-05"
# # model_name = "gemini/gemini-2.0-flash"
# lm = dspy.LM(
#     model=model_name,
#     max_tokens=65535,
#     # allowed_openai_params=["thinking"],
#     # thinking={"type": "enabled", "budget_tokens": 1024},
# )
# dspy.configure(lm=lm)

In [36]:
# from typing import Literal

# class QueryGenerationSignature(dspy.Signature):
#     """
#     Given a claim and its original context, generate a question where the claim is the direct answer.
#     The style of the question is determined by the ambiguity_level.

#     - If 'straightforward', the question should be direct and factual, often starting with Who, What, When, or Where. The claim should be the most obvious and concise answer.
#     - If 'ambiguous', the question should be more open-ended, subjective, or analytical, perhaps starting with Why, How, or asking for significance/implications. The claim should still be a valid and strong answer, but other interpretations might be possible.
#     - Do not use questions like "What would be an inaccurate way to ..." or "What is a false statement regarding..."
#     """

#     context: str = dspy.InputField(
#         desc="The source text where the claim originates, providing necessary context."
#     )
#     claim: str = dspy.InputField(
#         desc="The specific statement that should be the answer to the generated question."
#     )
#     ambiguity_level: str = dspy.InputField(
#         desc="Controls the style of the generated query. Options: 'straightforward' or 'ambiguous'."
#     )
#     query: str = dspy.OutputField(
#         desc="The generated question for which the claim is the intended answer."
#     )


# class QueryGenerator(dspy.Module):
#     """A module to generate a query from a claim and context."""
#     def __init__(self):
#         super().__init__()
#         self.query_generator = dspy.ChainOfThought(QueryGenerationSignature)

#     async def forward(self, claim: str, context: str, ambiguity: Literal["straightforward", "ambiguous"]) -> dspy.Prediction:
#         """
#         Generates a query from a claim and its context.

#         Args:
#             claim: The statement that should be the answer to the query.
#             context: The original text providing context for the claim.
#             ambiguity: The desired ambiguity level of the query.
#                       Must be either 'straightforward' or 'more ambiguous'.

#         Returns:
#             A dspy.Prediction object containing the generated 'query'.
#         """
#         if ambiguity not in ["straightforward", "ambiguous"]:
#             raise ValueError("ambiguity must be 'straightforward' or 'ambiguous'")
            
#         query_generator_async = dspy.asyncify(self.query_generator)

#         prediction = await query_generator_async(
#             claim=claim,
#             context=context,
#             ambiguity_level=ambiguity
#         )
#         return prediction

In [37]:
# ! pip install async-lru --quiet

In [38]:
# from tqdm.asyncio import tqdm
# from async_lru import alru_cache

# query_generator_module = QueryGenerator()

# semaphore = asyncio.Semaphore(10)

# @alru_cache(maxsize=None)
# async def generate_query(claim: str, context: str, ambiguity: str) -> str:
#     async with semaphore:
#         query = await query_generator_module.forward(
#             claim=claim,
#             context=context,
#             ambiguity=ambiguity
#         )

#         return query

# ambiguity = "straightforward"
# queries = await tqdm.gather(*[
#         generate_query(
#             claim=claim["claim"], 
#             context=claim["context"], 
#             ambiguity=ambiguity) 
#             for claim in claims_cached[:10]
#         ]
#     )


In [39]:
# claims_cached[0]

In [40]:
semaphore = asyncio.Semaphore(10)
results = await asyncio.gather(*[eval_generation_async(
    question=claim.query, 
    model_reply=claim.claim, 
    ground_truth=claim.context, 
    semaphore=semaphore) 
    for claim in claims]
    )

In [41]:
import pandas as pd

pd.DataFrame([dict(question=claim.query, 
                   model_reply=claim.claim,
                   ground_truth=claim.context,
                   score=result.score,
                   reason=result.reason) 
              for (claim, result) in zip(claims, results)])

,question,model_reply,ground_truth,score,reason
0,What are the pillars of Blue Ridge Outfitters'...,Blue Ridge Outfitters' differentiation strateg...,This differentiation is built upon pillars of ...,5.0,The generated response accurately identifies a...
1,Who are the competitors of Blue Ridge Outfitte...,"Based on the provided text, Blue Ridge Outfitt...","In the competitive outdoor retail landscape, B...",5.0,The generated response accurately identifies D...
2,What is the goal of Blue Ridge Outfitters' str...,The goal of Blue Ridge Outfitters' strategy is...,"By combining deep product knowledge, regional ...",4.5,The generated response accurately identifies t...
3,What is the name of the training program for B...,The name of the comprehensive 40-hour training...,Unlike the generalized training often found in...,5.0,The generated response accurately and complete...
4,How many hours is the 'Basecamp Training' prog...,Blue Ridge Outfitters mandates a comprehensive...,Unlike the generalized training often found in...,5.0,The generated response accurately and complete...
...,...,...,...,...,...
1501,What is being leveraged as part of the integra...,Blue Ridge Outfitters' integrated strategy lev...,The Guided Adventure Travel division serves as...,0.0,The generated response is completely inaccurat...
1502,What division of Blue Ridge Outfitters serves ...,"The Pathfinder Gear division, which develops n...",The Guided Adventure Travel division serves as...,0.0,The generated response incorrectly identifies ...
1503,How do the guided trips contribute to product ...,The Guided Adventure Travel division's trips a...,The Guided Adventure Travel division serves as...,0.5,The generated response is largely inaccurate. ...
1504,What is the long-term vision for the company's...,The long-term vision for Blue Ridge Outfitters...,* **Near-Term Focus (3-5 Years):** Strategic...,0.5,The generated response incorrectly states that...


In [42]:
import dotenv
from google.adk.runners import InMemoryRunner
from google.genai.types import Part
from google.genai.types import UserContent
from llm_auditor.agent import root_agent, critic_agent, llm_auditor
import textwrap

In [52]:
auditor_runner = InMemoryRunner(agent=llm_auditor)

def create_verification_prompt(claim: str) -> str:
    return f"Verify this claim: {claim}"

@alru_cache(maxsize=None)
async def revise_claim(claim: str) -> str:
    """
    Revises a claim using a runner session.

    Args:
        claim: The claim string to be evaluated.

    Returns:
        The rewritten claim.
    """
    session = auditor_runner.session_service.create_session(
        app_name=auditor_runner.app_name, user_id="test_user"
    )
    content = UserContent(parts=[Part(text=create_verification_prompt(claim))])
    events = []
    async for event in auditor_runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=content
    ):
        events.append(event)

    raw_text = events[-1].content.parts[0].text

    return raw_text

    # return prompt.CriticOutput.model_validate_json(raw_text)

In [53]:
dotenv.load_dotenv()

True

In [54]:
# response = await revise_claim(claim=claims[0].claim)
# response

In [55]:
from tqdm.asyncio import tqdm

async def _revise_single_claim(claim: Claim, semaphore: asyncio.Semaphore) -> str | None:
    """
    Revises a single claim using the 'revise_claim' async function.
    Includes concurrency control and basic error handling.

    Args:
        claim: The Claim object to evaluate.
        semaphore: An asyncio.Semaphore to limit concurrent access.

    Returns:
        The revised claim str, or None if an error occurs.
    """
    async with semaphore:
        try:
            # Call the actual evaluation function
            result = await revise_claim(claim=claim.claim)
            return result
        except Exception as e:
            print(f"An error occurred during evaluation of '{claim.claim}': {e}")
            return None

async def revise_claims_async(claims: list[Claim], max_concurrency: int = 5) -> list[str | None]:
    """
    Asynchronously evaluates a list of Claim objects with concurrency control
    and a progress bar.

    Args:
        claims: A list of Claim dataclass instances to evaluate.
        max_concurrency: The maximum number of concurrent evaluation tasks.

    Returns:
        A list of revised claim str's, or None if the revision failed for that claim.
    """
    semaphore = asyncio.Semaphore(max_concurrency)
    tasks = [_revise_single_claim(claim, semaphore) for claim in claims]

    print(f"\nStarting asynchronous evaluation of {len(claims)} claims (max concurrency: {max_concurrency})...")
    # Use tqdm.asyncio.tqdm.gather for concurrent execution with a progress bar
    results = await tqdm.gather(*tasks, desc="Revising Claims")
    print("Asynchronous revision complete.")
    return results

In [58]:
revised_claims = await revise_claims_async(claims=claims)

assert len(revised_claims) == len(claims)


Starting asynchronous evaluation of 1506 claims (max concurrency: 5)...


An error occurred during evaluation of 'The employee break area at the Blue Ridge Outfitters corporate headquarters offers views of the adjacent e-commerce distribution center.': 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}


An error occurred during evaluation of 'Blue Ridge Outfitters offers guided experiences in various domestic and international locations. Within the United States, trips are based near Asheville, North Carolina; Denver, Colorado; and Jackson, Wyoming. International expeditions are conducted in the European Alps (France and Switzerland), South America (Peru, and Patagonia in Chile/Argentina), Africa (Tanzania), and the Himalayas (Nepal).': 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}


Revising Claims: 100%|██████████| 1506/1506 [16:43<00:00,  1.50it/s]

Asynchronous revision complete.


In [59]:
semaphore = asyncio.Semaphore(10)
revised_results = await asyncio.gather(*[eval_generation_async(
    question=claim.query, 
    model_reply=revised_claim, 
    ground_truth=claim.context, 
    semaphore=semaphore) 
    for claim, revised_claim in zip(claims, revised_claims)]
    )

In [60]:
df_results = pd.DataFrame([dict(question=claim.query, 
                   answer=claim.claim,
                   context=claim.context,
                   score=result.score,
                   reason=result.reason,
                   revised_answer=revised_claim,
                   revised_score=revised_result.score,
                   revised_reason=revised_result.reason,
                   ) 
              for (claim, result, revised_claim, revised_result) in zip(claims, results, revised_claims, revised_results)])

In [61]:
df_results.head()

,question,answer,context,score,reason,revised_answer,revised_score,revised_reason
0,What are the pillars of Blue Ridge Outfitters'...,Blue Ridge Outfitters' differentiation strateg...,This differentiation is built upon pillars of ...,5.0,The generated response accurately identifies a...,Blue Ridge Outfitters' differentiation strateg...,5.0,The generated response accurately identifies a...
1,Who are the competitors of Blue Ridge Outfitte...,"Based on the provided text, Blue Ridge Outfitt...","In the competitive outdoor retail landscape, B...",5.0,The generated response accurately identifies D...,"Based on the provided text, Blue Ridge Outfitt...",5.0,The generated response accurately identifies t...
2,What is the goal of Blue Ridge Outfitters' str...,The goal of Blue Ridge Outfitters' strategy is...,"By combining deep product knowledge, regional ...",4.5,The generated response accurately identifies t...,The goal of Blue Ridge Outfitters' strategy is...,5.0,The generated response accurately and comprehe...
3,What is the name of the training program for B...,The name of the comprehensive 40-hour training...,Unlike the generalized training often found in...,5.0,The generated response accurately and complete...,The comprehensive 40-hour training program man...,5.0,The generated response accurately identifies t...
4,How many hours is the 'Basecamp Training' prog...,Blue Ridge Outfitters mandates a comprehensive...,Unlike the generalized training often found in...,5.0,The generated response accurately and complete...,"Yes, Blue Ridge Outfitters requires all new re...",5.0,The generated response accurately states that ...


score
5.0    625
0.0    421
1.0    184
4.5     97
0.5     88
2.0     31
4.0     20
2.5     13
3.0     10
3.5      9
1.5      5
4.8      3
Name: count, dtype: int64